<a href="https://colab.research.google.com/github/ecarreram-blip/se-alesysistemas/blob/main/Trading%20proyecto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==========================================
# CELDA 1: INSTALACIÓN E IMPORTACIÓN DE LIBRERÍAS
# ==========================================

# 1. Instalamos las herramientas necesarias
!pip install flask pyngrok pandas ta yfinance

# 2. Importamos las librerías al entorno
from flask import Flask, jsonify, request
from pyngrok import ngrok
import pandas as pd
import yfinance as yf
import ta
import threading
import time

print("✅ CELDA 1 COMPLETADA: Librerías instaladas e importadas correctamente.")

  Preparing metadata (setup.py) ... done
  Created wheel for ta: filename=ta-0.11.0-py3-none-any.whl size=29412 sha256=9575d96fd14c698df8ffbb8334869fa10bfd348748355bb1276677133cc83a1c
  Stored in directory: /root/.cache/pip/wheels/5c/a1/5f/c6b85a7d9452057be4ce68a8e45d77ba34234a6d46581777c6
Successfully built ta
✅ CELDA 1 COMPLETADA: Librerías instaladas e importadas correctamente.


In [2]:
# ==========================================
# CELDA 2: CONFIGURACIÓN Y OBTENCIÓN DE DATOS
# ==========================================

# 1. Tu token de Ngrok (No lo cambies)
NGROK_TOKEN = "3AiMRZqt7U0IN95wN8OXOTr6dcT_5txqfi3Hs4bQ3ntZU4Vbz"

# 2. Inicializamos la aplicación web
app = Flask(__name__)

# 3. Función para descargar los datos del Oro (XAU/USD) en tiempo real
def get_data():
    try:
        # Intenta descargar XAUUSD
        df = yf.download(tickers="XAUUSD=X", period="5d", interval="1m", progress=False)

        # Si falla, intenta con el futuro del oro (GC=F)
        if df is None or df.empty:
            df = yf.download(tickers="GC=F", period="5d", interval="1m", progress=False)

        if df is None or df.empty: return None

        # Ajustar el formato de las columnas para que el código lo entienda
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.droplevel(1)

        df.rename(columns={'Open':'open', 'High':'high', 'Low':'low', 'Close':'close', 'Volume':'volume'}, inplace=True)
        df.dropna(inplace=True)

        return df
    except Exception as e:
        print(f"Error descargando datos: {e}")
        return None

print("✅ CELDA 2 COMPLETADA: Motor de datos y configuración creados correctamente.")

✅ CELDA 2 COMPLETADA: Motor de datos y configuración creados correctamente.


In [3]:
# ==========================================
# CELDA 3: ANÁLISIS TÉCNICO Y MATEMÁTICO
# ==========================================

def analyze(user_entry=None, base_lot=0.01):
    try:
        df = get_data()
        if df is None or df.empty:
            return {"error": "Esperando datos..."}

        # Extraemos los precios
        close = df["close"]
        high = df["high"]
        low = df["low"]
        current_price = close.iloc[-1]

        # Volatilidad (ATR)
        current_atr = ta.volatility.AverageTrueRange(high, low, close).average_true_range().iloc[-1]

        # Definimos el precio base (si pusiste uno manual, usa ese; si no, usa el del mercado)
        base_price = user_entry if user_entry is not None else current_price

        # Indicadores
        rsi = ta.momentum.RSIIndicator(close, window=14).rsi()
        ema9 = ta.trend.EMAIndicator(close, window=9).ema_indicator()
        ema21 = ta.trend.EMAIndicator(close, window=21).ema_indicator()

        h1_high = high.tail(60).max()
        h1_low = low.tail(60).min()

        score_buy = 0
        score_sell = 0

        # Sistema de Puntuación
        if rsi.iloc[-1] < 35: score_buy += 25
        if rsi.iloc[-1] > 65: score_sell += 25

        if ema9.iloc[-1] > ema21.iloc[-1]: score_buy += 30
        else: score_sell += 30

        if current_price <= h1_low + 1: score_buy += 20
        if current_price >= h1_high - 1: score_sell += 20

        if close.iloc[-1] > close.iloc[-5]: score_buy += 10
        else: score_sell += 10

        # Definir dirección y calcular zonas
        if score_buy > score_sell:
            signal = "BUY"
            probability = score_buy
            tp1 = base_price + current_atr * 0.5
            tp2 = base_price + current_atr * 1.0
            tp3 = base_price + current_atr * 1.5
            re_entry1 = base_price - current_atr * 0.8
            re_entry2 = base_price - current_atr * 1.6
        else:
            signal = "SELL"
            probability = score_sell
            tp1 = base_price - current_atr * 0.5
            tp2 = base_price - current_atr * 1.0
            tp3 = base_price - current_atr * 1.5
            re_entry1 = base_price + current_atr * 0.8
            re_entry2 = base_price + current_atr * 1.6

        # Entregar los resultados
        return {
            "price": round(current_price, 2),
            "base_price": round(base_price, 2),
            "signal": signal,
            "probability": probability,
            "tp1": round(tp1, 2), "tp2": round(tp2, 2), "tp3": round(tp3, 2),
            "re_entry1": round(re_entry1, 2), "re_entry2": round(re_entry2, 2),
            "lot_contra1": round(base_lot * 2, 2), "lot_contra2": round(base_lot * 4, 2)
        }
    except Exception as e:
        print(f"Error en el análisis: {e}")
        return {"error": "Error calculando"}

print("✅ CELDA 3 COMPLETADA: Cerebro matemático configurado correctamente.")

✅ CELDA 3 COMPLETADA: Cerebro matemático configurado correctamente.


In [4]:
# ==========================================
# CELDA 4: CREACIÓN DE LA PÁGINA WEB (INTERFAZ)
# ==========================================

# 1. Configuración Anti-Caché para que tu celular siempre cargue lo más nuevo
@app.after_request
def add_header(response):
    response.headers['Cache-Control'] = 'no-store, no-cache, must-revalidate, post-check=0, pre-check=0, max-age=0'
    response.headers['Pragma'] = 'no-cache'
    response.headers['Expires'] = '-1'
    return response

# 2. Ruta de datos (donde el celular consulta los cálculos matemáticos)
@app.route("/data")
def data():
    user_entry_str = request.args.get('entry')
    user_lot_str = request.args.get('lot')

    user_entry = float(user_entry_str) if user_entry_str else None
    base_lot = float(user_lot_str) if user_lot_str else 0.01

    return jsonify(analyze(user_entry, base_lot))

# 3. Ruta principal (La página web que tú ves)
@app.route("/")
def home():
    return """
    <html>
    <head>
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <meta http-equiv="Cache-Control" content="no-cache, no-store, must-revalidate" />
    <meta http-equiv="Pragma" content="no-cache" />
    <meta http-equiv="Expires" content="0" />
    <title>XAU/USD MANAGER V6.2</title>
    <style>
        body { font-family: Arial; background: #0f0f0f; color: white; text-align: center; margin: 0; padding: 10px;}
        .box { background: #1e1e1e; padding: 15px; margin: 10px auto; width: 90%; max-width: 380px; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.3); box-sizing: border-box;}
        .grid-box { display: flex; justify-content: space-around; background: #1e1e1e; padding: 15px; margin: 10px auto; width: 90%; max-width: 380px; border-radius: 10px; box-sizing: border-box;}
        .buy { color: #00ff9d; font-size: 24px;}
        .sell { color: #ff4d4d; font-size: 24px;}
        .tp { color: #4da6ff; }
        .contra { color: #ffcc00; }
        h3 { margin-bottom: 5px; font-size: 14px; color: #aaa; text-transform: uppercase;}
        h2 { margin-top: 5px; margin-bottom: 5px;}
        .input-group { margin-bottom: 10px; }
        input[type="number"] { padding: 8px; border-radius: 5px; border: 1px solid #444; width: 140px; text-align: center; font-size: 14px; font-weight: bold; background: #2a2a2a; color: white; }
        button { padding: 8px 15px; border-radius: 5px; border: none; font-weight: bold; cursor: pointer; margin: 5px; transition: 0.2s;}
        button:hover { opacity: 0.8; }
        .btn-calc { background: #00ff9d; color: black; }
        .btn-clear { background: #ff4d4d; color: white; }
        .btn-update-manual { background: #4da6ff; color: white; width: 90%; max-width: 350px; padding: 12px; font-size: 16px; margin-top: 5px; border-radius: 8px;}
        .lot-badge { background: #333; padding: 3px 6px; border-radius: 4px; font-size: 12px; color: #fff; margin-left: 10px;}
        .update-text { font-size: 11px; color: #666; margin-top: 15px; }
    </style>
    </head>
    <body>
        <h2 style="color: #4da6ff; font-size: 20px;">XAU/USD MANAGER V6.2</h2>

        <div class="box" style="padding: 10px; height: 320px;">
            <div id="tv_chart" style="height: 100%; width: 100%;"></div>
        </div>

        <div class="box" style="border: 1px solid #333;">
            <div class="input-group">
                <h3>Mi Entrada Manual</h3>
                <input type="number" id="my_entry" step="0.01" placeholder="Ej: 2150.50">
            </div>
            <div class="input-group">
                <h3>Lote Inicial (Base)</h3>
                <input type="number" id="my_lot" step="0.01" value="0.01">
            </div>
            <button class="btn-calc" onclick="setValues()">Fijar Datos</button>
            <button class="btn-clear" onclick="clearValues()">Limpiar</button>
        </div>

        <div class="box">
            <h3>Precio de Mercado Actual</h3>
            <h2 id="price">...</h2>
            <p style="font-size: 12px; color: #888; margin: 0;">Usando como base: <strong id="base_price_display" style="color:#fff;">Mercado</strong></p>
        </div>

        <div class="grid-box">
            <div>
                <h3>Tendencia</h3>
                <h2 id="signal">...</h2>
            </div>
            <div>
                <h3>Fuerza</h3>
                <h2 id="prob">...</h2>
            </div>
        </div>

        <div class="box">
            <h3 class="tp">🎯 Toma de Ganancias</h3>
            <p id="tps" style="line-height: 1.8;">Calculando...</p>
        </div>

        <div class="box">
            <h3 class="contra">⚠️ Zonas para Meter Contra</h3>
            <p id="re_entries" style="line-height: 1.8;">Calculando...</p>
        </div>

        <button class="btn-update-manual" id="btn_manual" onclick="forceUpdate()">🔄 ACTUALIZAR AHORA</button>

        <p class="update-text">Última actualización: <span id="last_update_time">--:--:--</span></p>

        <script type="text/javascript" src="https://s3.tradingview.com/tv.js"></script>
        <script type="text/javascript">
            setTimeout(function() {
                new TradingView.widget({
                    "autosize": true,
                    "symbol": "OANDA:XAUUSD",
                    "interval": "1",
                    "timezone": "Etc/UTC",
                    "theme": "dark",
                    "style": "1",
                    "locale": "es",
                    "enable_publishing": false,
                    "backgroundColor": "#1e1e1e",
                    "hide_top_toolbar": true,
                    "hide_legend": true,
                    "save_image": false,
                    "container_id": "tv_chart"
                });
            }, 1000);
        </script>

        <script>
            let customEntry = null;

            function setValues() {
                let entryVal = document.getElementById("my_entry").value;
                if(entryVal) { customEntry = parseFloat(entryVal); }
                update();
            }

            function clearValues() {
                document.getElementById("my_entry").value = "";
                document.getElementById("my_lot").value = "0.01";
                customEntry = null;
                update();
            }

            function forceUpdate() {
                let btn = document.getElementById("btn_manual");
                btn.innerText = "⏳ Cargando...";
                update().then(() => { btn.innerText = "🔄 ACTUALIZAR AHORA"; });
            }

            async function update() {
                try {
                    let now = new Date();
                    document.getElementById("last_update_time").innerText = now.toLocaleTimeString();

                    let url = '/data';
                    let lotVal = document.getElementById("my_lot").value || 0.01;

                    let params = new URLSearchParams({ lot: lotVal });
                    if (customEntry !== null) { params.append('entry', customEntry); }

                    let res = await fetch(url + '?' + params.toString());
                    let d = await res.json();

                    if(d.error) return;

                    document.getElementById("price").innerText = d.price;
                    document.getElementById("base_price_display").innerText = customEntry ? d.base_price + " (Tu Entrada)" : "Precio Actual";

                    let s = document.getElementById("signal");
                    s.innerText = d.signal;
                    s.className = d.signal == "BUY" ? "buy" : "sell";

                    document.getElementById("prob").innerText = d.probability + "%";

                    document.getElementById("tps").innerHTML =
                        "TP1 (Corto): <b class='tp'>" + d.tp1 + "</b> <br> " +
                        "TP2 (Medio): <b class='tp'>" + d.tp2 + "</b> <br> " +
                        "TP3 (Largo): <b class='tp'>" + d.tp3 + "</b>";

                    document.getElementById("re_entries").innerHTML =
                        "Contra 1: <b class='contra'>" + d.re_entry1 + "</b> <span class='lot-badge'>Lote: " + d.lot_contra1 + "</span><br>" +
                        "Contra 2: <b class='contra'>" + d.re_entry2 + "</b> <span class='lot-badge'>Lote: " + d.lot_contra2 + "</span>";

                } catch(e) { console.log("Actualizando..."); }
            }

            setInterval(update, 10000);
            update();
        </script>
    </body>
    </html>
    """

print("✅ CELDA 4 COMPLETADA: Interfaz web y botón de actualización listos.")

✅ CELDA 4 COMPLETADA: Interfaz web y botón de actualización listos.


In [5]:
# ==========================================
# CELDA 5: ENCENDIDO DEL SERVIDOR Y ENLACE FINAL
# ==========================================

# 1. Apagamos cualquier conexión vieja para evitar errores
ngrok.kill()

# 2. Función para arrancar la web
def run():
    app.run(use_reloader=False)

# 3. Encendemos el servidor en segundo plano
threading.Thread(target=run).start()
time.sleep(2) # Esperamos 2 segundos a que arranque bien

# 4. Conectamos con Ngrok usando tu token
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(5000)

# 5. Entregamos el enlace
print("\n" + "="*50)
print("🚀 SERVIDOR EN LÍNEA - GESTOR V6.2 🚀")
print("👇 HAZ CLIC EN EL ENLACE AZUL DE AQUÍ ABAJO 👇")
print(public_url)
print("="*50)

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit



🚀 SERVIDOR EN LÍNEA - GESTOR V6.2 🚀
👇 HAZ CLIC EN EL ENLACE AZUL DE AQUÍ ABAJO 👇
NgrokTunnel: "https://gnarly-calcicolous-taneka.ngrok-free.dev" -> "http://localhost:5000"
